In [ ]:
epoch_in_minutes = 5

In [9]:
from ortools.sat.python import cp_model

def flexible_jobshop():
    model = cp_model.CpModel()

    # Problem data
    num_jobs = 3
    num_machines = 3

    # Each job is a list of operations.
    # Each operation is a list of (machine_id, duration) pairs.
    # jobs_data = [
    #     [[(0, 3), (1, 2)], [(0, 2)]],         # Job 0: 2 ops
    #     [[(1, 2)], [(0, 4), (1, 3)]]          # Job 1: 2 ops
    # ]
    
    jobs_data = [  # task = (machine_id, processing_time)
        [  # Job 0
            [(0, 3), (1, 1), (2, 5)],  # task 0
            [(0, 2), (1, 4), (2, 6)],  # task 1
            [(0, 2), (1, 3), (2, 1)],  # task 2
        ],
        [  # Job 1
            [(0, 2), (1, 3), (2, 4)],
            [(0, 1), (1, 5), (2, 4)],
            [(0, 2), (1, 1), (2, 4)],
        ],
        [  # Job 2
            [(0, 2), (1, 1), (2, 4)],
            [(0, 2), (1, 3), (2, 4)],
            [(0, 3), (1, 1), (2, 5)],
        ],
    ]
    jobs_arrival_epoch = [7, 8, 15]

    # Compute a rough upper bound on time horizon
    horizon = 0
    for job_index, job in enumerate(jobs_data):
        for task in job:
            max_task_duration = 0
            for alternative in task:
                max_task_duration = max(max_task_duration, alternative[1] + jobs_arrival_epoch[job_index])
            horizon += max_task_duration

    all_tasks = {}
    all_machines = [[] for _ in range(num_machines)]

    # Create variables
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            for m_id, duration in alternatives:
                suffix = f'_{job_id}_{task_id}_{m_id}'
                start = model.NewIntVar(0, horizon, 'start' + suffix)
                end = model.NewIntVar(0, horizon, 'end' + suffix)
                presence = model.NewBoolVar('presence' + suffix)
                interval = model.NewOptionalIntervalVar(start, duration, end, presence, 'interval' + suffix)

                all_tasks[(job_id, task_id, m_id)] = (start, end, interval, presence)
                all_machines[m_id].append((start, duration, interval))

    # Add constraints
    for job_id, job in enumerate(jobs_data):
        for task_id, alternatives in enumerate(job):
            # Only one machine must be selected for each operation
            presences = [all_tasks[(job_id, task_id, m_id)][3] for m_id, _ in alternatives]
            model.AddExactlyOne(presences)

            # Precedence constraints between operations
            if task_id > 0:
                prev_alts = jobs_data[job_id][task_id - 1]
                curr_alts = alternatives
                for prev_m_id, _ in prev_alts:
                    for curr_m_id, _ in curr_alts:
                        prev_end = all_tasks[(job_id, task_id - 1, prev_m_id)][1]
                        curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                        prev_presence = all_tasks[(job_id, task_id - 1, prev_m_id)][3]
                        curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                        model.Add(curr_start >= prev_end).OnlyEnforceIf([prev_presence, curr_presence])
            # first task's start time must be after arrival time
            if task_id == 0:
                curr_alts = alternatives
                for curr_m_id, _ in curr_alts:
                    curr_presence = all_tasks[(job_id, task_id, curr_m_id)][3]
                    curr_start = all_tasks[(job_id, task_id, curr_m_id)][0]
                    model.Add(curr_start >= jobs_arrival_epoch[job_id]).OnlyEnforceIf(curr_presence)
            
            

    # Machine capacity constraints: no overlapping intervals
    for machine_id in range(num_machines):
        machine_tasks = all_machines[machine_id]
        model.AddNoOverlap([interval for _, _, interval in machine_tasks]) # For every pair of intervals in the list, the solver ensures that if both are present, then their execution windows do not overlap

    # Objective: minimize makespan (latest end time)
    all_ends = [end for (_, end, _, _) in all_tasks.values()]
    makespan = model.NewIntVar(0, horizon, 'makespan')
    model.AddMaxEquality(makespan, all_ends)
    model.Minimize(makespan)

    # Solve model
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    # Output solution
    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        print(f'Minimized makespan: {solver.Value(makespan)}')
        for (j, t, m), (start, end, interval, presence) in all_tasks.items():
            if solver.Value(presence):
                print(f'Job {j}, Task {t} on Machine {m} → Start: {solver.Value(start)}, End: {solver.Value(end)}')
    else:
        print("No solution found.")

# Run the example
flexible_jobshop()

Minimized makespan: 19
Job 0, Task 0 on Machine 1 → Start: 7, End: 8
Job 0, Task 1 on Machine 1 → Start: 8, End: 12
Job 0, Task 2 on Machine 0 → Start: 13, End: 15
Job 1, Task 0 on Machine 0 → Start: 8, End: 10
Job 1, Task 1 on Machine 0 → Start: 10, End: 11
Job 1, Task 2 on Machine 0 → Start: 11, End: 13
Job 2, Task 0 on Machine 1 → Start: 15, End: 16
Job 2, Task 1 on Machine 0 → Start: 16, End: 18
Job 2, Task 2 on Machine 1 → Start: 18, End: 19
